# 网站抓取 + OpenAI 摘要

## 练习目标（理念）

把 **网页抓取** 和 **Chat Completions** 串成一条流水线：

1. 用本地模块 `scrape_website` 拉取目标页正文
2. 拼好 `system` / `user` 两条消息（含抓到的文本）
3. 调用 OpenAI 模型生成结构化摘要，并在笔记本里用 Markdown 展示

对应课程 **第 1 周 Day 1**：Website → Prompt → LLM → 可读摘要。

## 怎么跑

1. 确保同目录（或 PYTHONPATH）能导入 `selenium_scrapping.scrape_website`，且本机 Selenium 环境可用
2. `.env` 里配置好 `OPENAI_API_KEY`
3. 从上到下运行；可改 `url_to_scrape` 与 `user_prompt` 里的任务描述


In [ ]:
# ========== 导入：环境、抓取、展示、OpenAI 客户端 ==========

# 标准库 os：后面用 os.getenv 读 API Key
import os
# load_dotenv：把 .env 中的密钥加载进环境变量，避免写死在代码里
from dotenv import load_dotenv
# 本地抓取模块：selenium_scrapping.scrape_website（模块名按原 import 保留）
from selenium_scrapping import scrape_website
# IPython 展示：把模型返回的 Markdown 漂亮地渲染出来
from IPython.display import Markdown, display
# OpenAI 官方 Python SDK 客户端
from openai import OpenAI


In [ ]:
# ========== 环境：加载 .env 并取出 OPENAI_API_KEY ==========

# override=True：已有同名环境变量时，仍以 .env 覆盖
load_dotenv(override=True)
# 读出密钥字符串；后面 OpenAI() 默认也会从同名环境变量取
api_key = os.getenv('OPENAI_API_KEY')


In [ ]:
# ========== Prompt：system 定角色，user 定任务（发给模型的英文原文不翻译）==========

# system：要求按网站相似结构做摘要，并去掉特殊字符 / HTML 标签
system_prompt = """
You are an assistant that summarizes the content of a website. use the similar structure of the website to summarize the content.
remove any special characters and html tags.
"""
# user：具体业务问题模板；后面会把抓取到的网页正文拼在这段后面
user_prompt = """
Summarize how can enable my ai workforce and build ai native products or experiences from the text below:
Remove any special characters and html tags.

"""


In [ ]:
# ========== 抓取网页 + 组装 messages ==========

# 目标 URL：OpenAI Business 落地页（字符串保持原样，改 URL 即换站点）
url_to_scrape = "https://openai.com/business/"
# 调用 Selenium 抓取函数，得到页面文本（或清洗后的正文）
url_text = scrape_website(url_to_scrape)
# Chat Completions 的 messages：system 定规则，user = 任务说明 + 网页正文
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt + url_text}
]


In [ ]:
# ========== 调用模型并展示摘要 ==========

# 创建客户端：默认从环境变量 OPENAI_API_KEY 取密钥
openai = OpenAI()
# 非流式一次生成：模型 id 保持 gpt-4.1-nano；messages 来自上格
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 取出第一条 choice 的文本内容，用 Markdown 渲染到笔记本输出区
display(Markdown(response.choices[0].message.content))
